# Training pipeline audit

This notebook is the executable audit companion to `train.py`. It verifies the model-ready Level 3 artifact, approved feature selection, chronological splitting, and baseline model execution without changing the production training implementation.

**Holdout policy:** the default cells read only `FEATURE_RESEARCH_SEASONS` (2023–2024). They do not load or score `HOLDOUT_SEASON` (2025). A final holdout evaluation should happen only after the feature registry, workload model, model parameters, and evaluation protocol are frozen.

This notebook complements—not replaces—the automated tests in `tests/test_train.py`. Run it from the repository research environment (`pip install -e ".[research,dev]"`).

In [1]:
import hashlib
import importlib.util
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the repository root regardless of the notebook's launch directory.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "Python").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "Python").exists():
    raise RuntimeError("Could not locate repository root containing src/Python")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from Python import config
from Python.features import (
    LABEL_COLUMNS,
    MODEL_METADATA_COLUMNS,
    TARGET,
    model_feature_names,
)

# Import the real training module so this notebook audits production functions
# instead of maintaining a second implementation.
train_path = ROOT / "Models" / "Strikeout-Model" / "train.py"
spec = importlib.util.spec_from_file_location("strikeout_train", train_path)
if spec is None or spec.loader is None:
    raise RuntimeError(f"Could not load {train_path}")
train_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(train_module)

print("Training module:", train_path)
print("Level 3 artifact:", config.PITCHER_TRAINING_PATH)
print("Research seasons:", config.FEATURE_RESEARCH_SEASONS)
print("Protected holdout:", config.HOLDOUT_SEASON)

Training module: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\Models\Strikeout-Model\train.py
Level 3 artifact: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\Data\processed\pitcher_training.parquet
Research seasons: (2023, 2024)
Protected holdout: 2025


In [2]:
# Read only development seasons at the parquet scan boundary. Do not call
# train_module.load_frame() here because that function intentionally loads the
# complete production training artifact, including the protected holdout.
frame = pd.read_parquet(
    config.PITCHER_TRAINING_PATH,
    filters=[("season", "in", list(config.FEATURE_RESEARCH_SEASONS))],
)
frame["game_date"] = pd.to_datetime(frame["game_date"])
frame = (
    frame.dropna(subset=[TARGET, "game_date"])
    .sort_values(["game_date", "player_name"])
    .reset_index(drop=True)
)

# Normalize NumPy scalar values to native Python ints before JSON serialization.
observed_seasons = tuple(int(season) for season in sorted(frame["season"].unique()))
assert observed_seasons == config.FEATURE_RESEARCH_SEASONS
assert config.HOLDOUT_SEASON not in observed_seasons
assert not frame.duplicated(["game_pk", "pitcher"]).any()
assert frame["game_date"].is_monotonic_increasing
assert frame[TARGET].between(0, 1).all()

features = list(model_feature_names(frame))
selected = set(features)
assert not (selected & LABEL_COLUMNS)
assert not (selected & MODEL_METADATA_COLUMNS)
assert frame[features].select_dtypes(exclude=["number", "bool"]).empty

feature_hash = hashlib.sha256("\n".join(features).encode("utf-8")).hexdigest()
audit = {
    "rows": int(len(frame)),
    "columns": int(frame.shape[1]),
    "approved_features": int(len(features)),
    "date_range": [str(frame["game_date"].min().date()), str(frame["game_date"].max().date())],
    "seasons": list(observed_seasons),
    "ordered_feature_sha256": feature_hash,
}
print(json.dumps(audit, indent=2))

missingness = (
    frame[features].isna().mean().sort_values(ascending=False).rename("missing_rate")
)
missingness.head(15).to_frame()

{
  "rows": 9374,
  "columns": 271,
  "approved_features": 256,
  "date_range": [
    "2023-03-30",
    "2024-09-30"
  ],
  "seasons": [
    2023,
    2024
  ],
  "ordered_feature_sha256": "9df87cff28526d67aef41e8d19b92d44b81533dff6877d4f2ae3a17d79efad15"
}

,missing_rate
fs_vaa_P3,0.839983
fs_hb_P3,0.839983
fs_ivb_P3,0.839983
fs_velo_P3,0.839983
fs_spinrate_P3,0.839983
fs_ivb_P5,0.835929
fs_velo_P5,0.835929
fs_spinrate_P5,0.835929
fs_hb_P5,0.835929
fs_vaa_P5,0.835929


In [3]:
# Smoke-test the production split and model builders on development data only.
# This is a reproducibility audit, not the nested walk-forward feature-selection
# protocol and not a final holdout evaluation.
train, validation, test = train_module.chronological_split(frame)
partitions = {"train": train, "validation": validation, "test": test}

ranges = pd.DataFrame(
    {
        name: {
            "rows": len(part),
            "start": part["game_date"].min().date(),
            "end": part["game_date"].max().date(),
        }
        for name, part in partitions.items()
    }
).T
assert train["game_date"].max() < validation["game_date"].min()
assert validation["game_date"].max() < test["game_date"].min()
assert sum(len(part) for part in partitions.values()) == len(frame)
print(ranges)

rows = []
for model_name in ("mean", "ridge"):
    model = train_module.build_model(model_name)
    model.fit(train[features], train[TARGET])
    for split_name, part in (("validation", validation), ("test", test)):
        prediction = np.clip(model.predict(part[features]), 0, 1)
        rows.append(
            {
                "model": model_name,
                "split": split_name,
                **train_module.metrics(part[TARGET], prediction),
            }
        )

results = pd.DataFrame(rows)
assert np.isfinite(results[["mae", "rmse", "r2"]].to_numpy()).all()
results

            rows       start         end
train       6557  2023-03-30  2024-06-08
validation  1404  2024-06-09  2024-08-05
test        1413  2024-08-06  2024-09-30


,model,split,mae,rmse,r2
0,mean,validation,0.083741,0.104969,-0.001848
1,mean,test,0.085375,0.106955,-0.000975
2,ridge,validation,0.078575,0.099013,0.108619
3,ridge,test,0.078771,0.099272,0.137662


## Optional LightGBM smoke test

The next cell can exercise the production LightGBM builder and early-stopping API on development seasons only. It is disabled by default because some Windows Jupyter kernels can crash inside LightGBM's native library even when the same training call succeeds from the activated command-line environment.

Set `RUN_LIGHTGBM_SMOKE_TEST = True` only when explicitly testing that integration. The validation partition selects the boosting iteration, so its score is diagnostic rather than an unbiased performance estimate.

In [4]:
RUN_LIGHTGBM_SMOKE_TEST = False

if RUN_LIGHTGBM_SMOKE_TEST:
    required_state = ("train_module", "train", "validation", "test", "features")
    missing_state = [name for name in required_state if name not in globals()]
    if missing_state:
        raise RuntimeError(
            "Run the notebook from the first cell before the LightGBM smoke test; "
            f"missing state: {missing_state}"
        )
    if train_module.lgb is None:
        raise ImportError(
            'Install the research environment before this cell: pip install -e ".[research,dev]"'
        )
    lightgbm = train_module.build_model("lightgbm")
    lightgbm.fit(
        train_module.lightgbm_matrix(train, features),
        np.ascontiguousarray(train[TARGET].to_numpy(dtype=np.float64)),
        eval_X=train_module.lightgbm_matrix(validation, features),
        eval_y=np.ascontiguousarray(validation[TARGET].to_numpy(dtype=np.float64)),
        callbacks=[
            train_module.lgb.early_stopping(200),
            train_module.lgb.log_evaluation(50),
        ],
    )
    lightgbm_prediction = np.clip(
        lightgbm.predict(train_module.lightgbm_matrix(test, features)), 0, 1
    )
    lightgbm_report = {
        "best_iteration": int(lightgbm.best_iteration_),
        "development_test": train_module.metrics(test[TARGET], lightgbm_prediction),
    }
    print(json.dumps(lightgbm_report, indent=2))